In [5]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Barbra\proyecto\proyecto_final_modulo4\data\transactions_data.csv", nrows=5000)

# Financial Transactions Analysis

This notebook presents the data cleaning and transformation process applied to a financial transactions dataset. The goal is to prepare the data for analysis by ensuring consistency, creating new features, and enriching it with meaningful business categories.

In [6]:
df.info()
df.head()
df.tail()
df.shape
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              5000 non-null   int64  
 1   date            5000 non-null   str    
 2   client_id       5000 non-null   int64  
 3   card_id         5000 non-null   int64  
 4   amount          5000 non-null   str    
 5   use_chip        5000 non-null   str    
 6   merchant_id     5000 non-null   int64  
 7   merchant_city   5000 non-null   str    
 8   merchant_state  4365 non-null   str    
 9   zip             4356 non-null   float64
 10  mcc             5000 non-null   int64  
 11  errors          74 non-null     str    
dtypes: float64(1), int64(5), str(6)
memory usage: 468.9 KB


Index(['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors'],
      dtype='str')

## Data Cleaning

In this step, the dataset is cleaned to ensure data quality. This includes:
- Converting the transaction amount to numeric format  
- Parsing dates into datetime format  
- Removing unnecessary columns  
- Handling missing values  

LIMPIEZA

In [7]:
df["amount"] = df["amount"].replace('[\\$,]', '', regex=True).astype(float)
df["date"] = pd.to_datetime(df["date"])
df = df.drop(columns=["errors"])
df = df.dropna(subset=["amount", "date"])

## Feature Engineering

New variables are created to enhance the dataset:
- Transaction type (Income vs Expense)  
- Time-based features such as month and year  

VARIABLE

In [8]:
df["transaction_type"] = df["amount"].apply(lambda x: "Expense" if x < 0 else "Income")
df["month"] = df["date"].dt.to_period("M").astype(str)
df["year"] = df["date"].dt.year

## MCC Categorization

Merchant Category Codes (MCC) are mapped using an external reference dataset. This allows transactions to be classified into meaningful business categories, improving interpretability.

MCC

In [9]:
import json

with open("mcc_codes.json") as f:
    data = json.load(f)

mcc_df = pd.DataFrame(list(data.items()), columns=["mcc", "category"])
mcc_df["mcc"] = mcc_df["mcc"].astype(int)

df = df.merge(mcc_df, on="mcc", how="left")
df["category"] = df["category"].fillna("Other")

## Category Grouping

To simplify analysis, detailed categories are grouped into higher-level business categories. This reduces complexity and improves visualization.

AGRUPACION

In [10]:
df["category_group"] = df["category"].replace({
    "Grocery Stores, Supermarkets": "Food",
    "Eating Places and Restaurants": "Food",
    "Miscellaneous Food Stores": "Food",
    "Service Stations": "Transport",
    "Tolls and Bridge Fees": "Transport"
})

In [11]:
df.head()
df.columns

Index(['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'transaction_type', 'month', 'year', 'category', 'category_group'],
      dtype='str')

In [12]:
df["category_group"] = df["category_group"].fillna("Other")

In [14]:
df.to_csv("transactions_clean.csv", index=False)

## Final Dataset

The resulting dataset is clean, structured, and ready for analysis. It includes enriched categorical information and time-based features, making it suitable for SQL queries and visualization tools such as Power BI.